In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()

In [5]:
grids = dft.grid.Grids(mol)
grids.coords = coords = dat0["grid_coords"]
grids.weights = weights = dat0["grid_weights"]
ngrids = len(weights)

In [6]:
# Reference de_vxc from 06-1: this is what we want to reproduce.
de_vxc_ref = np.load("nh3_r_tpss0_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_vxc_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_vxc_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.831082156811122


In [7]:
ao = ni.eval_ao(mol, grids.coords, deriv=3)
# ao_t = ao.swapaxes(-1, -2) # transposed AO, [t, u, g]
rho = ni.eval_rho2(mol, ao, mo_coeff, mo_occ, xctype="MGGA")
rho = rho[[0, 1, 2, 3, 5]]
rho.shape

(5, 43328)

In [8]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="MGGA")
vxc = xc_eff[1]  # shape [5, ngrid]
fxc = xc_eff[2]  # shape [5, 5, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (5, 43328) fxc shape: (5, 5, 43328)


In [9]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9
XXX, XXY, XXZ, XYY, XYZ, XZZ = 10, 11, 12, 13, 14, 15
YYY, YYZ, YZZ, ZZZ = 16, 17, 18, 19

In [10]:
ao_dm0 = ao @ dm0
ao_dm0.shape

(20, 43328, 49)

In [13]:
drho = np.zeros((natm, 3, 5, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    # components
    DERIV_COMPONENTS = [
        # RHO part
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
        # SIGMA part (bra deriv 2)
        [(TX, X), (XX, O)],
        [(TX, Y), (XY, O)],
        [(TX, Z), (XZ, O)],
        [(TY, X), (YX, O)],
        [(TY, Y), (YY, O)],
        [(TY, Z), (YZ, O)],
        [(TZ, X), (ZX, O)],
        [(TZ, Y), (ZY, O)],
        [(TZ, Z), (ZZ, O)],
        # SIGMA part (bra deriv 1, ket deriv 1)
        [(TX, X), (X, X)],
        [(TX, Y), (X, Y)],
        [(TX, Z), (X, Z)],
        [(TY, X), (Y, X)],
        [(TY, Y), (Y, Y)],
        [(TY, Z), (Y, Z)],
        [(TZ, X), (Z, X)],
        [(TZ, Y), (Z, Y)],
        [(TZ, Z), (Z, Z)],
        # TAU part
        [(TX, 4), (XX, X)],
        [(TX, 4), (XY, Y)],
        [(TX, 4), (XZ, Z)],
        [(TY, 4), (YX, X)],
        [(TY, 4), (YY, Y)],
        [(TY, 4), (YZ, Z)],
        [(TZ, 4), (ZX, X)],
        [(TZ, 4), (ZY, Y)],
        [(TZ, 4), (ZZ, Z)],
    ];
    
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        drho[A, t, v] += np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
# scale symmetric coeff
drho[:, :, :4] *= 2

In [14]:
lib.fp(drho)

np.float64(16948168.18739754)

In [15]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))

-29.390069496788136


In [ ]:
dao_vxc_diag = np.zeros((3, 3, nao, nao))
dao_vxc = np.zeros((3, 3, nao, nao))

# --- dao_vxc --- #

DERIV_COMPONENTS = [
    # var, (comp A, comp B), (scale, comp bra, comp ket)
    # RHO part
    [0, (TX, TX), (2, X, X)],
    [0, (TX, TY), (2, X, Y)],
    [0, (TX, TZ), (2, X, Z)],
    [0, (TY, TX), (2, Y, X)],
    [0, (TY, TY), (2, Y, Y)],
    [0, (TY, TZ), (2, Y, Z)],
    [0, (TZ, TX), (2, Z, X)],
    [0, (TZ, TY), (2, Z, Y)],
    [0, (TZ, TZ), (2, Z, Z)],
    # SIGMA part (bra deriv 2)
    [1, (TX, TX), (2, XX, X)],
    [1, (TX, TX), (2, XX, X)],
]